# Part 1 — Model Development & Evaluation

**Goal:** score each transaction for money-laundering risk, and turn the score into
alerts within a fixed daily investigator budget.

Pipeline code lives in `src/aml_monitoring/`; this notebook is the narrative.

## 1. Setup

In [2]:
%load_ext autoreload
%autoreload 2

import time
import pandas as pd
from sklearn.metrics import average_precision_score, roc_auc_score

from aml_monitoring.config import TRAIN_MONTHS, VAL_MONTHS, TEST_MONTHS, ALERT_BUDGET_PCT
from aml_monitoring.dataset import load_or_build_features, make_dataset, FEATURE_COLS
from aml_monitoring.features.transaction import TRANSACTION_FEATURES
from aml_monitoring.features.entity import ENTITY_FEATURES
from aml_monitoring.models.train import train_baseline, train_lightgbm

pd.set_option("display.max_columns", None)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2. Load + build features

`load_or_build_features()` builds all 11 months of features once and caches them to
`artifacts/features.parquet` (a few minutes), then reads the cache on every later run.

Entity features are computed over the **whole** span, so val/test rows carry real
account history — exactly as they would at inference time.

In [3]:
t = time.time()
frame = load_or_build_features()
print(f"{frame.shape} in {time.time() - t:.0f}s")
frame[["timestamp", "month", "Amount", "Is_laundering"]].head()

(9504852, 35) in 6s


,timestamp,month,Amount,Is_laundering
0,2022-10-07 10:35:19,2022-10,1459.15,0
1,2022-10-07 10:35:20,2022-10,6019.64,0
2,2022-10-07 10:35:20,2022-10,14328.44,0
3,2022-10-07 10:35:21,2022-10,11895.00,0
4,2022-10-07 10:35:21,2022-10,115.25,0


## 3. Feature signal check

In [4]:
frame.groupby("Is_laundering")[TRANSACTION_FEATURES + ENTITY_FEATURES].mean().T.round(2)

Is_laundering,0,1
amount_log,8.35,8.34
amount_is_small,0.16,0.15
amount_is_large,0.00,0.03
cross_border,0.10,0.25
currency_mismatch,0.11,0.31
hour,14.19,13.84
day_of_week,2.97,2.93
sender_prior_txn_count,145.02,91.03
sender_secs_since_last,189698.58,1171892.27
sender_cnt_1d,25.31,0.69


## 4. Temporal split

Split by calendar month — no shuffling. Every training row precedes every validation
row, which precedes every test row. This mirrors production (train on the past,
predict the future) and prevents an account's future behaviour leaking backwards.

- **train**: 2022-10 … 2023-05 (8 months)
- **val**: 2023-06 — threshold selection only
- **test**: 2023-07 … 2023-08 — untouched until the end

In [5]:
ds = make_dataset(frame)
for name, y in [("train", ds.y_train), ("val", ds.y_val), ("test", ds.y_test)]:
    print(f"{name:5s} n={len(y):>9,}  positives={int(y.sum()):>5}  rate={y.mean():.5f}")
print(f"\nfeatures fed to the model: {len(FEATURE_COLS)}")

train n=7,048,788  positives= 7036  rate=0.00100
val   n=  897,243  positives= 1024  rate=0.00114
test  n=1,558,821  positives= 1813  rate=0.00116

features fed to the model: 26


## 5. Class imbalance

Prevalence is 0.10%. We handle it with **cost-weighting** (`scale_pos_weight`), not
resampling:

- SMOTE interpolates in a categorical + count feature space → fabricated, impossible rows
- The minority is 20+ typologies, not one cluster
- Synthetic rows have no timestamp → break the temporal split
- Rebalancing distorts calibration; we want honest ranking scores

Val/test stay at natural prevalence. The detection-vs-workload trade-off is a
**threshold** decision (Section 8), not a data decision. See ADR-0005.

Section 7 tests `scale_pos_weight` empirically.

## 6. Baseline — Logistic Regression

A transparent reference. If a linear model on these features already separates the
classes, the features carry real signal and any LightGBM gain is incremental.

In [6]:
baseline = train_baseline(ds)

2026/08/31 12:33:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/08/31 12:33:44 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[logreg] val PR-AUC = 0.0121


## 7. LightGBM + the imbalance sweep

We train LightGBM at four `scale_pos_weight` values. The naive choice is
negatives/positives (~1000). The sweep shows what that actually does to PR-AUC.
Each call logs its own MLflow run (`mlflow ui` to compare).

In [7]:
results, models = [], {}
for spw in [1.0, 5.0, 100.0, 1003.0]:
    m = train_lightgbm(ds, scale_pos_weight=spw)
    models[spw] = m
    p = m.predict_proba(ds.X_val)[:, 1]
    results.append({
        "scale_pos_weight": spw,
        "val_pr_auc": average_precision_score(ds.y_val, p),
        "val_roc_auc": roc_auc_score(ds.y_val, p),
    })

comparison = pd.DataFrame(results)
comparison

/Users/hafeez/Code/Enfuce-Case-Study/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's average_precision: 0.19955	valid_0's binary_logloss: 0.0111039


2026/08/31 12:33:58 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[lightgbm] scale_pos_weight=1.0  best_iter=1  val PR-AUC = 0.1833


/Users/hafeez/Code/Enfuce-Case-Study/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


...

/Users/hafeez/Code/Enfuce-Case-Study/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[2]	valid_0's average_precision: 0.0505929	valid_0's binary_logloss: 2.10324


2026/08/31 12:34:35 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


[lightgbm] scale_pos_weight=1003.0  best_iter=2  val PR-AUC = 0.0216


,scale_pos_weight,val_pr_auc,val_roc_auc
0,1.0,0.183272,0.759470
1,5.0,0.153802,0.774388
2,100.0,0.070585,0.849708
3,1003.0,0.021635,0.794750


In [8]:
# The model we take forward.
model = models[1.0]

## 8. Evaluation

*(next session)*

- PR-AUC vs ROC-AUC — why ROC-AUC misleads at 0.1% prevalence
- precision / recall / alerts-per-day at the 1% alert budget
- threshold selection with an explicit cost argument
- recall sliced by `Laundering_type`
- feature importance